# Laboratorio Integral: Auditoría de Calidad y Gobierno de Datos
**Caso de Estudio:** SECOP II y SECOP Integrado.

Este notebook implementa las métricas teóricas de calidad de datos para auditar la contratación pública. Exploraremos dimensiones como **Completitud, Concisión, Exactitud, Actualidad, Corrección, Consistencia y Trazabilidad**. Su objetivo es interpretar los hallazgos para formular políticas preventivas de Gobierno de Datos.

Diccionarios de datos:

[SECOP Integrado](https://www.datos.gov.co/api/views/rpmr-utcd/files/3484dabb-7bb3-4b4c-9b1a-990457db297d?download=true&filename=36_Diccionario_de_Datos-SECOP%20Integrado.docx)

[Ofertas por proceso](https://www.datos.gov.co/api/views/wi7w-2nvm/files/7df878b9-dfdd-405d-8d47-86ba8dca83e5?download=true&filename=20_Diccionario_de_Datos-%20SECOPII%20-%20Ofertas%20Por%20Proceso.docx)

Datos disponibles directo en datos.gov.co

[Ofertas](https://www.datos.gov.co/Estad-sticas-Nacionales/SECOPII-Ofertas-Por-Proceso/wi7w-2nvm/about_data) / 50 millones de filas aprox
[Integrado](https://www.datos.gov.co/Estad-sticas-Nacionales/SECOP-Integrado/rpmr-utcd/about_data) / 22 millones de filas aprox



In [0]:
import pandas as pd
from pyspark.sql.functions import col, when, isnull, lower, abs, count, sum as _sum, length, max as _max, min as _min

# URLs de Datos Abiertos Colombia
url_ofertas = "https://www.datos.gov.co/resource/wi7w-2nvm.csv?$limit=500000"
url_integrado = "https://www.datos.gov.co/resource/rpmr-utcd.csv?$limit=500000"

print("Descargando y cargando datos de Datos Abiertos Colombia...")
df_ofertas = spark.createDataFrame(pd.read_csv(url_ofertas, dtype=str))
df_integrado = spark.createDataFrame(pd.read_csv(url_integrado, dtype=str))

# Casteo numérico
df_ofertas = df_ofertas.withColumn("valor_de_la_oferta", col("valor_de_la_oferta").cast("double"))
df_integrado = df_integrado.withColumn("valor_contrato", col("valor_contrato").cast("double"))

display(df_ofertas.limit(5))

Sabemos que con 5000 filas podemos tener algunas estimaciones, pero necesitamos tener una mejor muestra para entender las cosas con mayor claridad

### Fase 1: Completitud y Concisión
1. **Completitud:** ¿La información es suficiente? (Mide datos faltantes).
2. **Concisión:** ¿La información es directa y sin redundancia? Evaluaremos si las descripciones de los procedimientos son innecesariamente largas, lo que dificulta el análisis automatizado.

In [0]:
# Métrica 1: Completitud (Búsqueda de Nulos)
print("--- Métrica: Completitud (Valores Nulos) ---")
display(df_ofertas.select([
    _sum(when(isnull(c) | (col(c) == 'nan') | ((df_ofertas.schema[c].dataType.typeName() == 'string') & (col(c) == '')), 1).otherwise(0)).alias(c) 
    for c in df_ofertas.columns
]))

# Métrica 2: Concisión (Longitud excesiva de textos)
print("--- Métrica: Concisión (Textos Extensos) ---")
df_concision = df_ofertas.withColumn("longitud_descripcion", length(col("descripcion_del_procedimiento")))
display(df_concision.filter(col("longitud_descripcion") > 100).select("id_del_proceso_de_compra", "longitud_descripcion", "descripcion_del_procedimiento"))

Cual es su opinión sobre estos resultados?, son buenos, malos?

> 🛑 **Punto de Decisión 1 (Completitud y Concisión):** 
>
> * **Completitud:** Si faltan datos en `nit_del_proveedor`, ¿cómo impacta esto la rendición de cuentas del Estado?
> * **Concisión:** Si las descripciones de los contratos son biblias de texto libre (>100 caracteres), ¿qué estándar propondría usted (ej. uso de catálogos estandarizados como UNSPSC) para mejorar la facilidad de análisis?, consideras que 100 caracteres está bien o la politica de calidad debería ser otra?

### Fase 2: Exactitud, Corrección y Actualidad
1. **Exactitud:** ¿Los datos reflejan la realidad correctamente? (Ej: Una oferta no debe valer $0 ni ser negativa).
2. **Corrección:** ¿Los datos están libres de errores? (Validaremos que el formato del NIT sea correcto).
3. **Actualidad:** ¿Los datos están actualizados? (Veremos el rango de fechas para saber si analizamos datos vigentes).

In [0]:
# Métrica 3: Exactitud (Valores lógicos)
df_exactitud = df_ofertas.filter((col("valor_de_la_oferta") <= 0) | isnull(col("valor_de_la_oferta")))

# Métrica 4: Corrección (Regex para formato de NIT)
df_correccion = df_ofertas.withColumn("nit_formato_valido", col("nit_del_proveedor").rlike("^[0-9]+-?[0-9]*$"))
df_invalidos = df_correccion.filter(col("nit_formato_valido") == False)

# Métrica 5: Actualidad (Rango temporal)
print("--- Métrica: Actualidad (Fechas de Registro) ---")
display(df_ofertas.select(_min("fecha_de_registro").alias("fecha_mas_antigua"), _max("fecha_de_registro").alias("fecha_mas_reciente")))

print(f"Alertas de Exactitud (Ofertas <= $0): {df_exactitud.count()}")
print(f"Alertas de Corrección (NIT mal formateado): {df_invalidos.count()}")

> 🛑 **Punto de Decisión 2 (Ciclo PDCA):** 
>
> Como responsable del Gobierno de Datos, redacte una política preventiva (Fase "Actuar" del ciclo PDCA) para la plataforma SECOP II que impida desde la interfaz gráfica que los proveedores ingresen NITs con letras (problema de *Corrección*) o valores de oferta en ceros (problema de *Exactitud*).

### Fase 3: Consistencia y Trazabilidad (Cruce de Fuentes)
1. **Consistencia:** ¿Los datos no se contradicen entre sistemas? (Cruce Ofertas vs SECOP Integrado).
2. **Trazabilidad:** ¿Se puede saber de dónde vienen los datos? (Auditar la existencia de la URL del contrato fuente).

In [0]:
# Cruce para validar Consistencia y Trazabilidad
df_cruce = df_ofertas.join(
    df_integrado,
    (df_ofertas["id_del_proceso_de_compra"] == df_integrado["numero_de_proceso"]) & 
    (df_ofertas["nit_del_proveedor"] == df_integrado["documento_proveedor"]),
    "inner"
)

# Métrica 6: Consistencia (Nombres idénticos)
df_consistencia = df_cruce.withColumn("inconsistencia_nombre", when(lower(col("nombre_proveedor")) != lower(col("nom_raz_social_contratista")), 1).otherwise(0))

# Métrica 7: Trazabilidad (Validar si existe el link del contrato)
df_trazabilidad = df_cruce.withColumn("falla_trazabilidad", when(isnull(col("url_contrato")) | (col("url_contrato") == ""), 1).otherwise(0))

display(df_consistencia.filter(col("inconsistencia_nombre") == 1).select("id_del_proceso_de_compra", "nombre_proveedor", "nom_raz_social_contratista", "url_contrato"))

### Reto Intermedio: Integridad Referencial (Registros Huérfanos)
Necesitamos saber qué procesos publicados en SECOP Integrado **no tienen ninguna oferta** registrada en el sistema de Ofertas. ¿Esto se consideraría un problema de calidad de datos, un problema del proceso de contratación (ej. licitación desierta), o ambos?

In [0]:
#Aquí el código

### Fase 4: Reto Estratégico - Diseño de una Nueva Métrica
Hasta ahora hemos validado métricas operativas (si el NIT tiene números, si no está vacío). Sin embargo, un NIT numérico podría ser inventado (Ej: "111111111"). 

**Su misión:** Diseñar una nueva métrica de calidad orientada al negocio que valide la existencia legal de las empresas ofertantes.

### Fase 5: Auditoría Forense y Modelos Estadísticos
El Gobierno de Datos no solo reacciona ante campos vacíos, sino que analiza el comportamiento de la información a gran escala para detectar fraudes, sesgos o errores sistemáticos. Utilizaremos dos modelos matemáticos sobre los valores de las ofertas y los proveedores.

#### Modelo 1: La Ley de Benford (Detección de Datos Inventados)
La Ley de Benford establece que, en conjuntos de datos numéricos naturales (como transacciones financieras o contratos), el primer dígito no se distribuye uniformemente. El número '1' debería aparecer como primer dígito aproximadamente el 30% de las veces, mientras que el '9' solo el 4.6%. Si los datos humanos son inventados o manipulados para evadir topes contractuales, la distribución real se alejará de la teórica.

In [0]:
from pyspark.sql.functions import substring, log10, round as _round

print("--- Auditoría Forense: Ley de Benford en Valores de Oferta ---")

# Filtramos valores válidos (mayores a 0)
df_validos = df_ofertas.filter(col("valor_de_la_oferta") > 0)

# Extraemos el primer dígito del valor de la oferta (casteando primero a entero largo para evitar decimales)
df_benford = df_validos.withColumn("primer_digito", substring(col("valor_de_la_oferta").cast("long").cast("string"), 1, 1))

# Calculamos la frecuencia real en nuestros datos
total_registros = df_benford.count()
df_distribucion = df_benford.filter(col("primer_digito").rlike("^[1-9]$")) \
    .groupBy("primer_digito").count() \
    .withColumn("frecuencia_real_pct", _round((col("count") / total_registros) * 100, 2))

# Calculamos la frecuencia teórica según la fórmula de Benford: log10(1 + 1/d) * 100
df_distribucion = df_distribucion \
    .withColumn("frecuencia_teorica_pct", _round(log10(1 + 1 / col("primer_digito").cast("double")) * 100, 2)) \
    .withColumn("desviacion_pct", abs(col("frecuencia_real_pct") - col("frecuencia_teorica_pct"))) \
    .orderBy("primer_digito")

display(df_distribucion)

#### Modelo 2: El Principio de Pareto (Regla del 80/20)
En el Gobierno de Datos, los recursos para limpiar y auditar información son limitados. El principio de Pareto nos sugiere que el 80% del valor del gasto público probablemente está concentrado en el 20% de los proveedores. Si logramos gobernar los datos de este pequeño grupo, habremos asegurado la mayor parte del impacto financiero.

In [0]:
from pyspark.sql.window import Window

print("--- Análisis de Concentración: Principio de Pareto ---")

# Calculamos el total ganado por cada proveedor
df_proveedores = df_ofertas.groupBy("nit_del_proveedor", "nombre_proveedor") \
    .agg(_sum("valor_de_la_oferta").alias("valor_total_adjudicado")) \
    .filter(col("valor_total_adjudicado") > 0)

# Calculamos el valor total de todas las ofertas
valor_total_estado = df_proveedores.agg(_sum("valor_total_adjudicado")).collect()[0][0]

# Ordenamos de mayor a menor y calculamos el porcentaje acumulado
window_spec = Window.orderBy(col("valor_total_adjudicado").desc())

df_pareto = df_proveedores \
    .withColumn("valor_acumulado", _sum("valor_total_adjudicado").over(window_spec)) \
    .withColumn("porcentaje_acumulado_gasto", _round((col("valor_acumulado") / valor_total_estado) * 100, 2))

# Mostramos el Top de proveedores
display(df_pareto.limit(20))

> 🛑 **Puntos de Decisión Estratégica (Modelos Analíticos):**
>
> **1. Sobre la Ley de Benford:** Compare la `frecuencia_real_pct` con la `frecuencia_teorica_pct`. Si usted nota que el dígito '4' o '5' tiene un pico inusual (una desviación muy alta), y usted sabe que la ley exige licitación pública para contratos mayores a $50.000.000, ¿Qué hipótesis de manipulación de datos o evasión de controles formularía?
>
> **2. Sobre el Principio de Pareto:** Si la tabla le demuestra que solo 15 proveedores concentran el 80% del presupuesto de la entidad, ¿Cómo rediseñaría su estrategia de **Data Stewardship** (gestión y cuidado de datos) para no desperdiciar tiempo limpiando los datos del 100% de los proveedores de forma manual?

---
### RESPUESTAS A LOS MODELOS ANALÍTICOS

**Hipótesis de Auditoría (Ley de Benford):**

[Su texto aquí...]

**Estrategia de Data Stewardship (Pareto):**

[Su texto aquí...]

---
### INFORME FINAL: SUSTENTACIÓN Y DECISIONES DE GOBIERNO

**1. Sustentación de Completitud y Concisión:**

[Redacte aquí su respuesta al Punto de Decisión 1...]

**2. Políticas Preventivas (Exactitud y Corrección):**

[Redacte aquí su respuesta al Punto de Decisión 2...]

**3. Política de Consistencia (Gestión de Datos Maestros - MDM):**
*Si el nombre del proveedor no coincide entre la oferta y el contrato firmado, ¿Qué política de arquitectura de datos implementaría para centralizar los nombres de los proveedores?*

[Su respuesta aquí...]

**4. Otras metricas recomendadas:**
*Que otras metricas recomendaría implementar para estos procesos de contratación? / Cómo las calcularía y por qué considera que serían utiles.*

[Su respuesta aquí...]

**5. RETO FINAL: Diseño de Métrica (Validación de NITs Reales):**
*Proponga y describa una nueva regla de calidad que permita a SECOP determinar si un NIT es una empresa jurídicamente "Real" y "Activa" al momento de ofertar. (Pista: Piense en interoperabilidad con otras entidades del Estado, como la DIAN o RUES, o en algoritmos de dígito de verificación).*

[Su respuesta aquí detallando la métrica, el nombre que le pondría, y cómo funcionaría el control]

---
**Instrucciones para la entrega:** Exporte este notebook en formato HTML (asegurándose de que las respuestas estén completas) y súbalo a la bloque neón.